In [0]:
%pip install mlflow==2.10.1 lxml==4.9.3 langchain==0.1.5 databricks-sdk==0.18.0 databricks-vectorsearch==0.22 cloudpickle==2.2.1 pydantic==2.5.2 sentence-transformers flashrank
%pip install pip mlflow[databricks]==2.10.1

dbutils.library.restartPython()

In [0]:
import time
import json

from pprint import pprint
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from mlflow.deployments import get_deploy_client
from flashrank import Ranker, RerankRequest

deploy_client = get_deploy_client("databricks")
vsc = VectorSearchClient(disable_notice = True)
ranker = Ranker(model_name = "rank-T5-flan", cache_dir =  "/Volumes/analytics/bronze/volume_da/GenAIStudy/flashrank") 

In [0]:
def endpoint_exists(vsc, vs_endpoint_name):
  """

  """
  try:
    return vs_endpoint_name in [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]
  except Exception as e:
    #Temp fix for potential REQUEST_LIMIT_EXCEEDED issue
    if "REQUEST_LIMIT_EXCEEDED" in str(e):
      print("WARN: couldn't get endpoint status due to REQUEST_LIMIT_EXCEEDED error. The demo will consider it exists")
      return True
    else:
      raise e

def wait_for_vs_endpoint_to_be_ready(vsc, vs_endpoint_name):
  """
  
  """
  for i in range(180):
    try:
      endpoint = vsc.get_endpoint(vs_endpoint_name)
    except Exception as e:
      #Temp fix for potential REQUEST_LIMIT_EXCEEDED issue
      if "REQUEST_LIMIT_EXCEEDED" in str(e):
        print("WARN: couldn't get endpoint status due to REQUEST_LIMIT_EXCEEDED error. Please manually check your endpoint status")
        return
      else:
        raise e
    status = endpoint.get("endpoint_status", endpoint.get("status"))["state"].upper()
    if "ONLINE" in status:
      return endpoint
    elif "PROVISIONING" in status or i <6:
      if i % 20 == 0: 
        print(f"Waiting for endpoint to be ready, this can take a few min... {endpoint}")
      time.sleep(10)
    else:
      raise Exception(f'''Error with the endpoint {vs_endpoint_name}. - this shouldn't happen: {endpoint}.\n Please delete it and re-run the previous cell: vsc.delete_endpoint("{vs_endpoint_name}")''')
  raise Exception(f"Timeout, your endpoint isn't ready yet: {vsc.get_endpoint(vs_endpoint_name)}")

def index_exists(vsc, endpoint_name, index_full_name):
    """
    
    """
    try:
        dict_vsindex = vsc.get_index(endpoint_name, index_full_name).describe()
        return dict_vsindex.get('status').get('ready', False)
    except Exception as e:
        if 'RESOURCE_DOES_NOT_EXIST' not in str(e):
            print(f'Unexpected error describing the index. This could be a permission issue.')
            raise e
    return False
    
def wait_for_index_to_be_ready(vsc, vs_endpoint_name, index_name):
  """
  
  """
  for i in range(180):
    idx = vsc.get_index(vs_endpoint_name, index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    
    if "ONLINE" in status:
      return
    if "UNKNOWN" in status:
      print(f"Can't get the status - will assume index is ready {idx} - url: {url}")
      return
    elif "PROVISIONING" in status:
      if i % 40 == 0: print(f"Waiting for index to be ready, this can take a few min... {index_status} - pipeline url:{url}")
      time.sleep(10)
    else:
        raise Exception(f'''Error with the index - this shouldn't happen. DLT pipeline might have been killed.\n Please delete it and re-run the previous cell: vsc.delete_index("{index_name}, {vs_endpoint_name}") \nIndex details: {idx}''')
  raise Exception(f"Timeout, your index isn't ready yet: {vsc.get_index(index_name, vs_endpoint_name)}")

def wait_for_index_to_be_deleted(vsc, vs_endpoint_name, index_name):
  """
  
  """ 
  try:
    idx = vsc.get_index(index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    
    if "ONLINE" in status:
      rebuild_index(vsc, vs_endpoint_name, index_name)

    else:
      print(f"O vs_index {index_name} foi deletado com sucesso!")
      return

  except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
      print(f"O vs_index {index_name} foi deletado com sucesso!")
      return  

def rebuild_index(vsc, vs_endpoint_name, index_name):
  """
  
  
  """
  vsc.delete_index(vs_endpoint_name, index_name)
  time.sleep(5)
  wait_for_index_to_be_deleted(vsc, vs_endpoint_name, index_name)

  try:
    #Trigger a sync to update our vs content with the new data saved in the table
    vsc.create_delta_sync_index(
      endpoint_name = VECTOR_SEARCH_ENDPOINT_NAME,
      index_name = vs_index_fullname,
      source_table_name = source_table_fullname,
      pipeline_type = "TRIGGERED",
      primary_key = "id",
      embedding_source_column = 'chunked_text', #The column containing our text
      embedding_model_endpoint_name = 'gpt-poc-solucoes-embedding' #The embedding endpoint used to create the embeddings
    )

  except Exception as e:
    if "currently pending deletion" in str(e):
      wait_for_index_to_be_deleted(vsc, vs_endpoint_name, index_name)
      rebuild_index(vsc, vs_endpoint_name, index_name)

In [0]:
vsc_endpoint = "eliel_paes_studies"
catalog = "analytics"
db_name = "bronze"
source_table = "pdf_text_embeddings"
vsc_index_name = f"{source_table}_vsc"

In [0]:
wait_for_vs_endpoint_to_be_ready(vsc, vsc_endpoint)

In [0]:
# defining the full name of the source table
source_table_full_name = f"{catalog}.{db_name}.{source_table}"

# defining the full name of the index
vsc_index_fullname = f"{catalog}.{db_name}.{vsc_index_name}"

# create or sync the index
if not index_exists(vsc, vsc_endpoint, vsc_index_fullname):
    print(f"Creating index {vsc_index_fullname} on endpoint {vsc_endpoint}...")
    vsc.create_delta_sync_index(
        endpoint_name = vsc_endpoint,
        index_name = vsc_index_fullname,
        source_table_name = source_table_full_name, 
        pipeline_type = "TRIGGERED", # Sync needs to be manually triggered. Can be done through the UI or api. For the API check the changedatafeed log of the source table and then, trigger the pipeline if necessary.
        primary_key = "id",
        embedding_vector_column = 'embeddings', # Here, a column containing the embeddings is provided. Another option is to provide the column to be used to calculate the embeddings. This is done by setting embedding_source_column and embedding_model_endpoint_name.        
        embedding_dimension = 1024
    )
else:
    vsc.get_index(vsc_endpoint, vsc_index_fullname).sync()

wait_for_index_to_be_ready(vsc, vsc_endpoint, vsc_index_fullname)

In [0]:
questions = json.load(open("../config/questions.json")) 

question = questions[0]["question"]
print(question)

# calculatin the embeddings for the question
response = deploy_client.predict(endpoint = "databricks-bge-large-en", inputs = {"input": [question]})
question_embeddings = [r["embedding"] for r in response["data"]]

# get similar 5 documents
results = vsc.get_index(vsc_endpoint, vsc_index_fullname).similarity_search(
    query_vector = question_embeddings[0],
    columns = ["pdf_name", "content"],
    num_results = 5
)

passages = []
for doc in results.get("result", {}).get("data_array", []):
    new_doc = {"file": doc[0], "text": doc[1]}
    passages.append(new_doc)

pprint(passages)

In [0]:
rerankrequest = RerankRequest(query = question, passages = passages)
reranking_results = ranker.rerank(rerankrequest)

print(*reranking_results, sep = "\n\n")    